# Przegląd slajdu z python-pptx

1. **Analog Selection Pane** – pętla `for` po wszystkich obiektach na slajdzie.
2. **Szukanie obiektu po nazwie** – `if shape.name == ...` i odczyt danych z wykresu pierścieniowego.

In [1]:
from pptx import Presentation

PLIK = "slajd_silownia_z_basenem.pptx"

prs = Presentation(PLIK)
slide = prs.slides[0]
print(f"{PLIK}: {len(prs.slides)} slajd(y), na pierwszym {len(slide.shapes)} obiektów")

slajd_silownia_z_basenem.pptx: 1 slajd(y), na pierwszym 16 obiektów


## 1. Selection Pane

`slide.shapes` zwraca obiekty w kolejności z-order (od spodu do wierzchu). PowerPoint w Selection Pane pokazuje je odwrotnie – na górze listy to, co leży najwyżej – dlatego iterujemy po `reversed(...)`.

In [2]:
EMU_NA_CAL = 914400  # python-pptx trzyma wymiary w EMU

print(f"{'nazwa':<20} {'typ':<12} {'x':>6} {'y':>6} {'szer':>6} {'wys':>6}  tekst")
print("-" * 80)
for shape in reversed(slide.shapes):
    typ = shape.shape_type.name if shape.shape_type else "?"
    tekst = shape.text_frame.text.replace("\n", " | ")[:30] if shape.has_text_frame else ""
    print(f"{shape.name:<20} {typ:<12} "
          f"{shape.left / EMU_NA_CAL:6.2f} {shape.top / EMU_NA_CAL:6.2f} "
          f"{shape.width / EMU_NA_CAL:6.2f} {shape.height / EMU_NA_CAL:6.2f}  {tekst}")

nazwa                typ               x      y   szer    wys  tekst
--------------------------------------------------------------------------------
Zrodlo               TEXT_BOX       0.60   7.00  12.10   0.30  Dane przykładowe (fikcyjne). S
Legenda_5            TEXT_BOX       9.60   6.57   2.90   0.30  Sauna  10%
Legenda_kolor_5      AUTO_SHAPE     9.30   6.63   0.16   0.16  
Legenda_4            TEXT_BOX       9.60   6.29   2.90   0.30  Basen  14%
Legenda_kolor_4      AUTO_SHAPE     9.30   6.35   0.16   0.16  
Legenda_3            TEXT_BOX       9.60   6.01   2.90   0.30  Zajęcia grupowe  18%
Legenda_kolor_3      AUTO_SHAPE     9.30   6.07   0.16   0.16  
Legenda_2            TEXT_BOX       9.60   5.73   2.90   0.30  Cardio i maszyny  24%
Legenda_kolor_2      AUTO_SHAPE     9.30   5.79   0.16   0.16  
Legenda_1            TEXT_BOX       9.60   5.45   2.90   0.30  Wolne ciężary  34%
Legenda_kolor_1      AUTO_SHAPE     9.30   5.51   0.16   0.16  
KPI_srodek           TEXT_BOX       9

## 2. Znajdź wykres po nazwie i odczytaj dane

Nazwa `Wykres_strefy` to ta sama, którą widać w Selection Pane (i którą nadał jej `generuj_slajd.py`).

In [3]:
SZUKANA_NAZWA = "Wykres_strefy"

for shape in slide.shapes:
    if shape.name == SZUKANA_NAZWA:
        wykres = shape.chart
        plot = wykres.plots[0]
        kategorie = list(plot.categories)

        print(f"Znaleziono: {shape.name}")
        print(f"Typ wykresu: {wykres.chart_type.name}")
        print(f"Kategorii: {len(kategorie)}, serii: {len(plot.series)}\n")

        for seria in plot.series:
            print(f"Seria: {seria.name}")
            for punkt, kategoria, wartosc in zip(seria.points, kategorie, seria.values):
                kolor = punkt.format.fill.fore_color.rgb  # kolor wycinka ustawiony w skrypcie
                print(f"  {kategoria:<18} {wartosc:>5.0f}%   #{kolor}")
            print(f"  {'suma':<18} {sum(seria.values):>5.0f}%")
        break
else:
    print(f"Nie ma obiektu o nazwie {SZUKANA_NAZWA!r}")

Znaleziono: Wykres_strefy
Typ wykresu: DOUGHNUT
Kategorii: 5, serii: 1

Seria: Udział
  Wolne ciężary         34%   #E4572E
  Cardio i maszyny      24%   #263238
  Zajęcia grupowe       18%   #F2A65A
  Basen                 14%   #3A8DB5
  Sauna                 10%   #9EA7AD
  suma                 100%
